In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

In [ ]:
with open("data/raw/characteristics_with_ind_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

## 划分训练集/测试集/验证集

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

## 线性回归模型（OLS）

In [ ]:
from sklearn.metrics import mean_squared_error
model = LinearRegression()
model.fit(X_train, y_train)

# 打印模型的系数和截距
print('Coefficients:', model.coef_)
print('Intercept:', model.intercept_)
y_test_pred = model.predict(X_test)

test_mse = mean_squared_error(y_test, y_test_pred)
print('Test MSE:', test_mse)

In [ ]:
from sklearn.metrics import r2_score
r2_test = r2_score(y_test, y_test_pred)
print('Test R2:', r2_test)

In [ ]:
import matplotlib.pyplot as plt
def Drawing(y_test, result, title):
    fig = plt.figure(figsize=(20, 3))
    axes = fig.add_subplot(1, 1, 1)
    line1, = axes.plot(range(len(y_test)),y_test,'b',label='Actual_Value')
    line2, = axes.plot(range(len(y_test_pred)),y_test_pred,'r--',label='Predicted',linewidth=2)
    axes.grid()
    fig.tight_layout()
    plt.legend(handles=[line1,line2])
    plt.title(title)
    plt.show()

In [ ]:
Drawing(y_test, y_test_pred,'Linear Regression')

## 随机森林

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
y_test_pred = rf_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
print('Test MSE:', test_mse)
print('Test R2:', test_r2)

## 梯度增强回归树（GBRT）

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
gbrt_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
# 训练模型
gbrt_model.fit(X_train, y_train)
# 在验证集和测试集上进行预测
y_val_pred = gbrt_model.predict(X_val)
y_test_pred = gbrt_model.predict(X_test)

# 计算并打印验证集和测试集的均方误差（MSE）和 R² 值
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)
print('Validation MSE:', val_mse)
print('Test MSE:', test_mse)
print('Validation R2:', val_r2)
print('Test R2:', test_r2)

## 弹性网（ENet）

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNet
ElasticNet_clf = ElasticNet(alpha=0.001,l1_ratio=0.71)
ElasticNet_clf.fit(X_train,y_train)
ElasticNet_clf_score = ElasticNet_clf.score(X_test,y_test)  #模型得分
ElasticNet_clf_result = ElasticNet_clf.predict(X_test)  #验证集对应的预测结果
print("特征向量 w = ",ElasticNet_clf.coef_)
print("偏置常数 b = ",ElasticNet_clf.intercept_)
print("迭代次数：",ElasticNet_clf.n_iter_)
print("拟合优度R^2：",ElasticNet_clf_score)
 
Drawing(y_test, ElasticNet_clf_result,'Ridge Regression')

## Lasso回归

In [ ]:
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=1.0,fit_intercept=True, precompute=False, max_iter=1000,  tol=0.0001, warm_start=False,  positive=False,selection='cyclic')
lasso_model.fit(X_train, y_train)
y_test_pred = lasso_model.predict(X_test)
Lasso_clf = Lasso(alpha=0.001)
Lasso_clf.fit(X_train,y_train)
Lasso_clf_score = Lasso_clf.score(X_test,y_test)  #模型得分
Lasso_clf_result = Lasso_clf.predict(X_test)  #验证集对应的预测结果
print("特征向量 w = ",Lasso_clf.coef_)
print("偏置常数 b = ",Lasso_clf.intercept_)
print("迭代次数：",Lasso_clf.n_iter_)
print("拟合优度R^2：",Lasso_clf_score)
 
Drawing(y_test, Lasso_clf_result,'Lasso Regression')

## NN1

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(1, activation='linear'))
model.compile(optimizer='adam', loss='mean_squared_error')

# 训练模型
model.fit(X_train, y_train, epochs=100, batch_size=32)
y_test_pred = model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)

test_r2 = r2_score(y_test, y_test_pred)
print('Test MSE:', test_mse)
print('Test R2:', test_r2)

## NN2

In [ ]:
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='linear'))

# 编译模型
model.compile(optimizer='adam', loss='mean_squared_error')

# 训练模型
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val))

# 在验证集和测试集上进行预测
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# 计算并打印验证集和测试集的均方误差（MSE）和 R² 值
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('Validation MSE:', val_mse)
print('Test MSE:', test_mse)
print('Validation R2:', val_r2)
print('Test R2:', test_r2)

## NN3

In [ ]:
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='linear'))

# 编译模型
model.compile(optimizer='adam', loss='mean_squared_error')

# 训练模型
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val))

# 在验证集和测试集上进行预测
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# 计算并打印验证集和测试集的均方误差（MSE）和 R² 值
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('Validation MSE:', val_mse)
print('Test MSE:', test_mse)
print('Validation R2:', val_r2)
print('Test R2:', test_r2)

## NN4

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import mean_squared_error, r2_score
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(4, activation='relu'))  # 第四个隐藏层
model.add(Dense(1, activation='linear'))  # 输出层
model.compile(optimizer='adam', loss='mean_squared_error')
checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss', mode='min')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 训练模型，使用检查点和早停机制
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

# 加载最佳模型权重
model.load_weights('best_model.h5')

# 在验证集和测试集上进行预测
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# 计算并打印验证集和测试集的均方误差（MSE）和 R² 值
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('Validation MSE:', val_mse)
print('Test MSE:', test_mse)
print('Validation R2:', val_r2)
print('Test R2:', test_r2)

## NN5

In [ ]:
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(4, activation='relu'))

# 添加第五个隐藏层，这里可以根据需要调整神经元数量
model.add(Dense(2, activation='relu'))

# 添加输出层，因为是回归问题，所以使用'linear'激活函数
model.add(Dense(1, activation='linear'))

# 编译模型
model.compile(optimizer='adam', loss='mean_squared_error')

# 训练模型
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val))

# 在验证集和测试集上进行预测
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# 计算并打印验证集和测试集的均方误差（MSE）和 R² 值
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('Validation MSE:', val_mse)
print('Test MSE:', test_mse)
print('Validation R2:', val_r2)
print('Test R2:', test_r2)

## 主成分回归

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
scaler = StandardScaler()
pca = PCA(n_components=3) 
regressor = LinearRegression()
pcr_pipeline = make_pipeline(scaler, pca, regressor)
pcr_pipeline.fit(X_train, y_train)
y_train_pred = pcr_pipeline.predict(X_train)
y_test_pred = pcr_pipeline.predict(X_test)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print('Train R2:', train_r2)
print('Test R2:', test_r2)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 假设X_train是训练数据集的特征矩阵

# 数据标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 进行PCA，选择三个主成分
pca = PCA(n_components=3)
X_train_pca = pca.fit_transform(X_train_scaled)

# 绘制三维得分图
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], X_train_pca[:, 2], c='pink', marker='o')
ax.set_title('3D PCA Scores Plot')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
plt.show()